# Demo 01 · Del score ML a una recomendación GenAI explicable

            Esta demo conecta directamente con los notebooks de analítica de ventas e interpretabilidad ML. Partimos del modelo de inactividad ya entrenado, seleccionamos un cliente con score alto y construimos una recomendación comercial **trazable**.

            No usamos una API de LLM. Simulamos la capa generativa con una plantilla controlada para que el foco didáctico sea la explicabilidad: qué datos entran, qué señales del modelo se usan, qué documentos respaldan la recomendación y qué cautelas se comunican.


In [1]:
from pathlib import Path
import sys
import json
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 160)
pd.set_option("display.width", 160)
pd.set_option("display.max_colwidth", 180)

def find_project_root(start=None):
    current = Path(start or Path.cwd()).resolve()
    for candidate in [current, *current.parents]:
        if (candidate / "datos").exists() and (candidate / "escenarios").exists():
            return candidate
    raise FileNotFoundError("No se ha encontrado la raíz del proyecto FraSoHome.")

PROJECT_ROOT = find_project_root()
DEMO_DIR = PROJECT_ROOT / "escenarios" / "09-genai-interpretabilidad"
ARTIFACT_DIR = DEMO_DIR / "artifacts"
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)
sys.path.insert(0, str(DEMO_DIR))

from common.genai_frasohome import (
    append_trace,
    build_agent_trace,
    build_rag_index,
    chunk_documents,
    evaluate_agent_trace,
    evaluate_rag_questions,
    grounded_answer,
    keyword_coverage,
    load_crm,
    load_knowledge_documents,
    load_sales_artifact,
    local_feature_contributions,
    recommendation_from_score,
    retrieve,
    simple_answer_rubric,
    summarize_customer_context,
)

print("PROJECT_ROOT:", PROJECT_ROOT)
print("DEMO_DIR:", DEMO_DIR)


PROJECT_ROOT: C:\repos\frasohome-demos
DEMO_DIR: C:\repos\frasohome-demos\escenarios\09-genai-interpretabilidad


## 1. Cargar el artefacto de ventas

            La demo depende del artefacto generado en la sesión anterior: datasets, pipeline, particiones y métricas. Esta dependencia es intencional: el asistente GenAI no inventa el análisis, lo consume.


In [2]:
artifact = load_sales_artifact(PROJECT_ROOT)
print("Artefacto cargado:", artifact["_path"])
print("Bloques disponibles:", [k for k in artifact.keys() if not k.startswith("_")])

churn = artifact["churn"]
display(pd.Series(churn["result"]["metrics"]).to_frame("métrica"))


Artefacto cargado: C:\repos\frasohome-demos\escenarios\04-analitica-ventas-ml\artifacts\sales_models_artifact.joblib
Bloques disponibles: ['created_at', 'source', 'churn', 'campaign']


,métrica
rows,596.000000
target_rate,0.473154
test_roc_auc,0.755515
test_average_precision,0.768213
test_f1,0.619048
test_accuracy,0.677852


## 2. Seleccionar un cliente prioritario y explicar el score

            En un sistema real podríamos usar SHAP, LIME o explicaciones precalculadas. Para esta demo usamos una aproximación local basada en señales transformadas e importancia del modelo, suficiente para enseñar el contrato de explicación.


In [3]:
score, row_raw, contributions = local_feature_contributions(churn["result"], top_n=12)
customer_context = summarize_customer_context(row_raw, contributions)

print(f"Score de inactividad seleccionado: {score:.3f}")
display(row_raw.T.head(30))
display(contributions)
display(pd.Series(customer_context).to_frame("valor"))


Score de inactividad seleccionado: 0.947


,593
frequency_docs,1
sales_lines,1
units_sold,1.0
monetary_gross,309.5
margin_gross,0.0
avg_discount_pct,0.0
pct_discounted_lines,0.0
avg_unit_price,309.5
n_categories,0
n_products,1


,feature,local_weighted_signal,abs_value
0,gross_margin_rate,0.037371,0.037371
1,margin_gross,0.032193,0.032193
2,ciudad_Córdoba,0.029915,0.029915
3,recency_days,0.025564,0.025564
4,avg_order_value,0.025056,0.025056
5,provincia_Córdoba,0.022182,0.022182
6,monetary_gross,0.021772,0.021772
7,avg_discount_pct,0.021545,0.021545
8,pct_discounted_lines,0.020585,0.020585
9,n_categories,0.019932,0.019932


,valor
cliente,cliente seleccionado
recency_days,149
frequency_docs,1
monetary_gross,309.5
tier,NaN
last_channel,POS
last_category,NaN
top_signals,"[gross_margin_rate, margin_gross, ciudad_Córdoba, recency_days, avg_order_value]"


## 3. Recuperar evidencia documental

            La recomendación comercial no debería apoyarse solo en el score. Consultamos la base de conocimiento de FraSoHome para traer reglas de fidelización, permisos y cautelas operativas.


In [4]:
documents = load_knowledge_documents(PROJECT_ROOT)
chunks = chunk_documents(documents)
rag_index = build_rag_index(chunks, include_obsolete=False, include_security_test=False)

query = "reglas CRM fidelización consentimiento marketing campaña reactivación cliente Platino cupón"
evidence = retrieve(rag_index, query, top_k=5)
display(evidence[["rank", "score", "file_name", "chunk_id", "area"]])


,rank,score,file_name,chunk_id,area
26,1,0.176492,FS-KB-07_Guia_Fidelizacion_CRM_v1.1.docx,FS-KB-07-CH000,05_CRM
27,2,0.099714,FS-KB-07_Guia_Fidelizacion_CRM_v1.1.docx,FS-KB-07-CH001,05_CRM
4,3,0.049220,FS-KB-01_Politica_Devoluciones_v1.3_Vigente.docx,FS-KB-01-CH004,01_Politicas
3,4,0.045375,FS-KB-01_Politica_Devoluciones_v1.3_Vigente.docx,FS-KB-01-CH003,01_Politicas
5,5,0.043682,FS-KB-01_Politica_Devoluciones_v1.3_Vigente.docx,FS-KB-01-CH005,01_Politicas


## 4. Generar una recomendación explicable

            La salida separa cuatro elementos: recomendación, evidencia, cautelas y trazabilidad. Esta separación es más importante que el estilo textual.


In [5]:
recommendation = recommendation_from_score(score, customer_context, evidence)
print(recommendation["recommendation"])
print("\nFuentes usadas:")
for source in recommendation["evidence_files"]:
    print("-", source)
print("\nCautelas:")
for caveat in recommendation["caveats"]:
    print("-", caveat)


Priorizar una acción de reactivación para cliente nan. El score de inactividad es 0.95; la recencia observada es 149 días. Personalizar el mensaje con novedades o ventaja moderada relacionada con nan. Antes de activar, comprobar consentimiento, presión comercial y margen esperado.

Fuentes usadas:
- FS-KB-07_Guia_Fidelizacion_CRM_v1.1.docx
- FS-KB-07_Guia_Fidelizacion_CRM_v1.1.docx
- FS-KB-01_Politica_Devoluciones_v1.3_Vigente.docx

Cautelas:
- El score es predictivo, no causal.
- La recomendación debe respetar consentimiento y política CRM.
- Conviene medir uplift o conversión incremental antes de escalar.


## 5. Trazabilidad de extremo a extremo

            Una recomendación GenAI debería poder reconstruirse después. La traza mínima registra datos, modelo, explicación, retrieval y respuesta final.


In [6]:
trace = [
    {"step": 1, "component": "modelo_ml", "detail": "score de inactividad", "value": round(score, 4)},
    {"step": 2, "component": "interpretabilidad_ml", "detail": "señales locales principales", "value": contributions["feature"].head(6).tolist()},
    {"step": 3, "component": "rag", "detail": "fuentes recuperadas", "value": evidence["file_name"].head(3).tolist()},
    {"step": 4, "component": "generación_controlada", "detail": "recomendación con cautelas", "value": recommendation["recommendation"]},
]
display(pd.DataFrame(trace))

trace_path = ARTIFACT_DIR / "demo_01_recommendation_trace.json"
trace_path.write_text(json.dumps(trace, ensure_ascii=False, indent=2), encoding="utf-8")
print("Traza guardada:", trace_path)


,step,component,detail,value
0,1,modelo_ml,score de inactividad,0.9469
1,2,interpretabilidad_ml,señales locales principales,"[gross_margin_rate, margin_gross, ciudad_Córdoba, recency_days, avg_order_value, provincia_Córdoba]"
2,3,rag,fuentes recuperadas,"[FS-KB-07_Guia_Fidelizacion_CRM_v1.1.docx, FS-KB-07_Guia_Fidelizacion_CRM_v1.1.docx, FS-KB-01_Politica_Devoluciones_v1.3_Vigente.docx]"
3,4,generación_controlada,recomendación con cautelas,Priorizar una acción de reactivación para cliente nan. El score de inactividad es 0.95; la recencia observada es 149 días. Personalizar el mensaje con novedades o ventaja moder...


Traza guardada: C:\repos\frasohome-demos\escenarios\09-genai-interpretabilidad\artifacts\demo_01_recommendation_trace.json


## 6. Lectura para el instructor

            El mensaje clave para clase: la IA generativa no sustituye a la interpretabilidad ML, la consume. Una buena recomendación separa lo que es dato, lo que es inferencia del modelo, lo que está respaldado por documentos y lo que es propuesta comercial.
